# Fact Part Usage — Gold Layer

Aggregated fact table at the grain of one row per part. Answers how popular and versatile each part is across sets, themes, and colors.

## What this notebook does

1. **Read** — Loads `fct_set_inventory` from gold.
2. **Transform** — Aggregates per part: total sets used in, total themes, total quantity, most common color, distinct colors available, and year range.
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/gold/delta_volume/fct_part_usage`.
4. **Register** — Creates the catalog table `lego_brickbase.gold.fct_part_usage`.

## Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, countDistinct, sum as _sum, min as _min, max as _max,
    row_number,
)

spark = SparkSession.builder.getOrCreate()

GOLD_FCT_SET_INVENTORY = "lego_brickbase.gold.fct_set_inventory"
GOLD_DIM_PART          = "lego_brickbase.gold.dim_part"
DELTA_TABLE_PATH       = "/Volumes/lego_brickbase/gold/delta_volume/fct_part_usage"
CATALOG_TABLE          = "lego_brickbase.gold.fct_part_usage"

## Read and Transform

Aggregate `fct_set_inventory` to the part grain and enrich with part dimension attributes.

In [ ]:
df_inv = spark.table(GOLD_FCT_SET_INVENTORY)
df_part = spark.table(GOLD_DIM_PART)

# Core metrics per part
df_part_agg = (
    df_inv
    .groupBy("part_key")
    .agg(
        countDistinct("set_key").alias("total_sets_used_in"),
        countDistinct("root_theme_name").alias("total_themes_used_in"),
        _sum("quantity").alias("total_quantity_across_sets"),
        countDistinct("color_key").alias("distinct_colors_available_in"),
        _min("year").alias("first_year_appeared"),
        _max("year").alias("latest_year_appeared"),
    )
)

# Most common color per part
w = Window.partitionBy("part_key").orderBy(col("color_qty").desc())
df_most_common_color = (
    df_inv
    .groupBy("part_key", "color_name")
    .agg(_sum("quantity").alias("color_qty"))
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .select(
        col("part_key").alias("_mc_part_key"),
        col("color_name").alias("most_common_color"),
    )
)

# Join with dim_part and most common color
df_fct = (
    df_part_agg
    .join(df_part, df_part_agg["part_key"] == df_part["part_key"], "left")
    .join(df_most_common_color, df_part_agg["part_key"] == df_most_common_color["_mc_part_key"], "left")
    .select(
        df_part_agg["part_key"],
        df_part["part_name"],
        df_part["part_category_name"],
        df_part["is_active"],
        col("total_sets_used_in"),
        col("total_themes_used_in"),
        col("total_quantity_across_sets"),
        col("most_common_color"),
        col("distinct_colors_available_in"),
        col("first_year_appeared"),
        col("latest_year_appeared"),
    )
)

df_fct.printSchema()
df_fct.display(10, truncate=False)

## Write to Delta Volume and Register Catalog Table

In [ ]:
(
    df_fct
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE}
    AS SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")
print(f"Catalog table ready: {CATALOG_TABLE}")